# Ders 3: ARMA Modelleri
- 3.1 ARMA(p,q) Süreçleri
- 3.2 ARMA(p,q) Sürecinin ACF ve PACF'si
- 3.3 ARMA Süreçlerinin Öngörüsü


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_process import arma_generate_sample
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from scipy.linalg import toeplitz
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
print("Hazır!")


## 3.1 ARMA(p, q) Süreçleri

**Tanım:** $\{X_t\}$ süreci ARMA(p,q) ise:
$$X_t - \phi_1 X_{t-1} - \cdots - \phi_p X_{t-p} = Z_t + \theta_1 Z_{t-1} + \cdots + \theta_q Z_{t-q}$$

Operatör notasyonuyla: $\phi(B)X_t = \theta(B)Z_t$

**Nedensellik Koşulu:** $\phi(z) = 1 - \phi_1 z - \cdots - \phi_p z^p \neq 0$ for $|z| \leq 1$

**Tersinirlik Koşulu:** $\theta(z) = 1 + \theta_1 z + \cdots + \theta_q z^q \neq 0$ for $|z| \leq 1$


In [ ]:
# ── Karakteristik Kökler ve Nedensellik/Tersinirlik Kontrolü ──

def check_causality_invertibility(ar_params, ma_params):
    '''ar_params: [φ₁, φ₂, ...] (operatörden sonra işaretler)'''
    # AR polinomu kökleri: φ(z) = 1 - φ₁z - φ₂z² - ...
    ar_poly = np.array([1] + [-p for p in ar_params])
    ar_roots = np.roots(ar_poly)
    
    ma_poly = np.array([1] + list(ma_params))
    ma_roots = np.roots(ma_poly)
    
    print(f"AR kökleri: {ar_roots}")
    print(f"  |AR kökleri|: {np.abs(ar_roots).round(4)}")
    print(f"  Nedensel mi? (tüm kökler |z|>1): {all(np.abs(ar_roots) > 1)}")
    print(f"MA kökleri: {ma_roots}")
    print(f"  |MA kökleri|: {np.abs(ma_roots).round(4)}")
    print(f"  Tersinir mi? (tüm kökler |z|>1): {all(np.abs(ma_roots) > 1)}")
    print()
    return ar_roots, ma_roots

models_to_check = [
    ("AR(1) φ=0.8",    [0.8],       []),
    ("AR(1) φ=1.2",    [1.2],       []),     # nedensel değil!
    ("AR(2) φ₁=0.5,φ₂=0.3", [0.5,0.3], []),
    ("MA(1) θ=0.6",    [],          [0.6]),
    ("MA(1) θ=1.5",    [],          [1.5]),  # tersinir değil!
    ("ARMA(1,1)",      [0.7],       [0.4]),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
theta = np.linspace(0, 2*np.pi, 300)
unit_circle_x = np.cos(theta)
unit_circle_y = np.sin(theta)

for idx, (name, ar, ma) in enumerate(models_to_check):
    ax = axes[idx//3, idx%3]
    ax.plot(unit_circle_x, unit_circle_y, 'k-', lw=1.5, label='Birim çember')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    
    print(f"{'='*45}")
    print(f"Model: {name}")
    ar_r, ma_r = check_causality_invertibility(ar, ma)
    
    if len(ar_r) > 0:
        ax.scatter(1/ar_r.real, 1/ar_r.imag if np.any(ar_r.imag != 0) else np.zeros(len(ar_r)),
                   marker='x', s=100, color='red', zorder=5, label='AR kökleri (1/z)')
    if len(ma_r) > 0:
        ax.scatter(1/ma_r.real, 1/ma_r.imag if np.any(ma_r.imag != 0) else np.zeros(len(ma_r)),
                   marker='o', s=100, color='blue', zorder=5, label='MA kökleri (1/z)')
    
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_aspect('equal'); ax.set_title(name, fontsize=10)
    ax.legend(fontsize=7)
    ax.fill_between(unit_circle_x, unit_circle_y, alpha=0.05, color='gray')

plt.tight_layout(); plt.show()


## 3.2 ACF ve PACF Hesaplama

### 3.2.1 Yule-Walker Denklemleri (AR süreci için)

AR(p) modeli için: $\phi(B)X_t = Z_t$

Yule-Walker denklemleri:
$$\rho(h) - \phi_1\rho(h-1) - \cdots - \phi_p\rho(h-p) = 0, \quad h > 0$$

Matris formunda: $\Gamma_p \boldsymbol{\phi} = \boldsymbol{\gamma}_p$

### 3.2.2 ARMA için Genel ACF

$|h| > q$ için: $\phi(B)\rho(h) = 0$, yani ACF AR polinomuna göre azalır.


In [ ]:
# ── Yule-Walker ile Teorik ACF Hesaplama ──

def ar_acf_yulewalker(phi_params, max_lag=30):
    '''AR(p) için Yule-Walker denklemlerini çözerek teorik ACF hesapla'''
    p = len(phi_params)
    phi = np.array(phi_params)
    
    # İlk p+1 ACF değerini Yule-Walker sistemiyle bul
    # [ρ(0), ρ(1), ..., ρ(p)] için Toeplitz sistemi
    # Matris A * rho = b
    A = np.zeros((p+1, p+1))
    A[0, 0] = 1
    for i in range(p):
        A[0, i+1] = -phi[i]
        
    for h in range(1, p+1):
        A[h, h] = 1
        for k in range(1, p+1):
            idx = abs(h - k)
            if idx <= p:
                A[h, k if k != h else 0] -= phi[k-1] if k != h else 0
    
    # Basit rekursif yöntem:
    rho = np.zeros(max_lag + 1)
    rho[0] = 1.0
    
    # Başlangıç değerleri için Yule-Walker sistemi
    Gamma = toeplitz([1.0] + [0.0]*(p-1))
    for i in range(p):
        for j in range(p):
            Gamma[i, j] = 0  # sıfırla
    
    # Daha basit: direkt rekursiyon
    # ρ(h) = φ₁ρ(h-1) + ... + φₚρ(h-p) for h > 0
    # ρ(0) = 1 (normalize)
    # İlk p değeri için: belirli koşullar
    
    # Numerik yaklaşım: uzun MA temsili
    from statsmodels.tsa.arima_process import arma_acovf
    ar = np.array([1] + [-p for p in phi_params])
    ma = np.array([1])
    acvf = arma_acovf(ar, ma, nobs=max_lag+1)
    rho = acvf / acvf[0]
    return rho

# Farklı AR modelleri karşılaştırma
ar_models = {
    'AR(1) φ=0.7':       [0.7],
    'AR(1) φ=-0.7':      [-0.7],
    'AR(2) φ₁=0.7,φ₂=-0.2': [0.7, -0.2],
    'AR(2) kompleks kökler': [1.0, -0.5],
}

fig, axes = plt.subplots(len(ar_models), 2, figsize=(14, 10))
lags = 25

for i, (name, phi) in enumerate(ar_models.items()):
    rho = ar_acf_yulewalker(phi, max_lag=lags)
    
    axes[i, 0].stem(range(lags+1), rho, linefmt='steelblue', 
                     markerfmt='o', basefmt='k-')
    axes[i, 0].axhline(0, color='k', lw=0.5)
    ci = 1.96 / np.sqrt(300)
    axes[i, 0].axhline(ci, color='red', ls='--', lw=0.8, alpha=0.7)
    axes[i, 0].axhline(-ci, color='red', ls='--', lw=0.8, alpha=0.7)
    axes[i, 0].set_title(f'{name} — Teorik ACF')
    axes[i, 0].set_xlabel('Gecikme h')
    
    # Simülasyon
    from statsmodels.tsa.arima_process import arma_generate_sample
    ar_poly = np.array([1] + [-p for p in phi])
    X = arma_generate_sample(ar_poly, [1], 300)
    plot_pacf(X, lags=20, ax=axes[i, 1], color='darkorange', title=f'{name} — Örneklem PACF')

plt.tight_layout(); plt.show()


## 3.2.4 Model Tanıma: ACF ve PACF Özet Tablosu

| Model | ACF | PACF |
|-------|-----|------|
| AR(p) | Üstel/sinüsoidal azalma | Gecikme p'den sonra kesilir |
| MA(q) | Gecikme q'dan sonra kesilir | Üstel/sinüsoidal azalma |
| ARMA(p,q) | İkisi de azalır | İkisi de azalır |


In [ ]:
# ── Model Tanıma Rehberi: Interaktif Karşılaştırma ──
np.random.seed(42)
n = 500

configs = [
    ('AR(1)', [1, -0.8], [1], 'ACF: üstel azalır, PACF: lag 1. gecikmede kesilir'),
    ('AR(2)', [1,-0.6,-0.3],[1], 'ACF: üstel/salınımlı, PACF: lag 2. gecikmede kesilir'),
    ('MA(1)', [1], [1, 0.7], 'ACF: lag 1. gecikmede kesilir, PACF: üstel azalır'),
    ('MA(2)', [1], [1,-0.5,0.3], 'ACF: lag 2. gecikmede kesilir, PACF: azalır'),
    ('ARMA(1,1)',[1,-0.7],[1,0.4],'ACF ve PACF: ikisi de azalır'),
    ('ARMA(2,1)',[1,-0.5,-0.2],[1,0.3],'ACF ve PACF: ikisi de azalır'),
]

fig, axes = plt.subplots(len(configs), 3, figsize=(16, 18))

for i, (name, ar, ma, desc) in enumerate(configs):
    X = arma_generate_sample(ar, ma, n)
    
    axes[i, 0].plot(X[:100], lw=0.9, color='steelblue')
    axes[i, 0].set_title(f'{name} — {desc}', fontsize=9)
    axes[i, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    plot_acf(X, lags=20, ax=axes[i, 1], color='steelblue', title='ACF')
    plot_pacf(X, lags=20, ax=axes[i, 2], color='darkorange', title='PACF', method='ywm')

plt.tight_layout(); plt.show()


## 3.3 ARMA Süreçlerinin Öngörüsü

ARMA(p,q) için $h$-adım öngörüsü:

$$\hat{X}_{n+h} = \phi_1 \hat{X}_{n+h-1} + \cdots + \phi_p \hat{X}_{n+h-p} + \hat{Z}_{n+h} + \theta_1 \hat{Z}_{n+h-1} + \cdots + \theta_q \hat{Z}_{n+h-q}$$

Burada $\hat{X}_j = X_j$ ($j \leq n$) ve $\hat{Z}_j = 0$ ($j > n$), $\hat{Z}_j = X_j - \hat{X}_j$ ($j \leq n$).

**Öngörü ortalaması:** $h \to \infty$ iken $\hat{X}_{n+h} \to \mu = 0$

**Öngörü varyansı:** $v(h) = \sigma^2 \sum_{j=0}^{h-1} \psi_j^2$ ($h \to \infty$ iken $\sigma^2 \sum_{j=0}^{\infty} \psi_j^2 = \gamma(0)$)


In [ ]:
# ── ARMA Öngörüsü ve Güven Aralıkları ──
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(10)
n_train = 150
n_forecast = 30

# ARMA(2,1) verisi oluştur
from statsmodels.tsa.arima_process import arma_generate_sample
X_full = arma_generate_sample([1, -0.6, -0.2], [1, 0.4], n_train + n_forecast, scale=1)
X_train = X_full[:n_train]
X_test  = X_full[n_train:]

# Model fit
model = ARIMA(X_train, order=(2, 0, 1))
fit   = model.fit()
print(fit.summary())

# Öngörü
forecast = fit.get_forecast(steps=n_forecast)
fc_mean  = forecast.predicted_mean
fc_ci    = forecast.conf_int(alpha=0.05)

# Grafik
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sol: Tüm seri + öngörü
t_all = np.arange(n_train + n_forecast)
axes[0].plot(t_all[:n_train], X_train, 'steelblue', lw=1.2, label='Eğitim verisi')
axes[0].plot(t_all[n_train:], X_test, 'green', lw=1.2, label='Gerçek değerler')
axes[0].plot(t_all[n_train:], fc_mean, 'r-', lw=2, label='ARMA(2,1) öngörü')
axes[0].fill_between(t_all[n_train:], fc_ci[:, 0], fc_ci[:, 1],
                      alpha=0.2, color='red', label='%95 GA')
axes[0].axvline(n_train, color='k', ls='--', lw=1)
axes[0].legend(fontsize=9); axes[0].set_title('ARMA(2,1) h-Adım Öngörüsü')

# Sağ: Öngörü belirsizliği
h_vals = np.arange(1, n_forecast+1)
# Teorik std: ψ ağırlıkları (AR→MA∞ dönüşümü)
# Basit: σ * sqrt(Σ ψⱼ²)
pred_std = np.array([forecast.se_mean[h] for h in range(n_forecast)])

axes[1].plot(h_vals, pred_std, 'r-o', ms=4, lw=1.5, label='Öngörü Std. Hatası')
axes[1].axhline(np.sqrt(X_train.var()), color='blue', ls='--', label='Süreç std (γ(0)^0.5)')
axes[1].set_xlabel('Öngörü ufku h'); axes[1].set_ylabel('Standart Hata')
axes[1].set_title('Öngörü Belirsizliği h ile Artıyor')
axes[1].legend()

plt.tight_layout(); plt.show()

# Hata metrikleri
from sklearn.metrics import mean_squared_error, mean_absolute_error
mse = mean_squared_error(X_test, fc_mean)
mae = mean_absolute_error(X_test, fc_mean)
print(f"\nÖngörü MSE:  {mse:.4f}")
print(f"Öngörü MAE:  {mae:.4f}")
print(f"Öngörü RMSE: {np.sqrt(mse):.4f}")


In [ ]:
# ── Öngörü Ufku ve Model Karşılaştırması ──
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(7)
n = 200

# AR(1) verisi
X = arma_generate_sample([1, -0.8], [1], n, scale=1)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col_idx, h in enumerate([1, 5, 20]):
    errors = []
    for start in range(50, n - h):
        model = ARIMA(X[:start], order=(1, 0, 0))
        fit = model.fit()
        fc = fit.forecast(steps=h)
        errors.append(X[start + h - 1] - (fc[-1] if hasattr(fc, "iloc") else fc[-1]))
    
    axes[0, col_idx].hist(errors, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
    axes[0, col_idx].axvline(0, color='red', ls='--')
    axes[0, col_idx].set_title(f'{h}-Adım Öngörü Hataları')
    axes[0, col_idx].set_xlabel('Hata')
    
    axes[1, col_idx].plot(errors, lw=0.7, color='steelblue')
    axes[1, col_idx].axhline(0, color='red', ls='--')
    axes[1, col_idx].axhline(np.std(errors), color='green', ls=':', label=f'σ={np.std(errors):.3f}')
    axes[1, col_idx].axhline(-np.std(errors), color='green', ls=':')
    axes[1, col_idx].set_title(f'{h}-Adım Hatası Zamanla'); axes[1, col_idx].legend()
    
    print(f"h={h:2d}: Ortalama Hata={np.mean(errors):.4f}, Std={np.std(errors):.4f}, RMSE={np.sqrt(np.mean(np.array(errors)**2)):.4f}")

plt.tight_layout(); plt.show()
print("\nGözlem: h artıkça öngörü belirsizliği artıyor!")


## Özet

| Kavram | Detay |
|--------|-------|
| **ARMA(p,q)** | $\phi(B)X_t = \theta(B)Z_t$ |
| **Nedensellik** | AR köklerinin birim çember dışında olması |
| **Tersinirlik** | MA köklerinin birim çember dışında olması |
| **ACF-PACF Tanıma** | AR↔PACF kesilir, MA↔ACF kesilir |
| **h-adım öngörü** | Belirsizlik h ile artar, uzak vadede ortalamaya döner |

